# Ablation: Judge Models

Compares Qwen-2.5-14B vs Llama-3.1-8B as LLM judges.
Measures inter-judge correlation and scoring bias.

In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, load_llm, create_summary_chain, create_judge_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt, get_judge_prompt
from sm_sip.pipelines import run_inference
from sm_sip.metrics.judge import llm_judge_evaluate
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory
import numpy as np

In [ ]:
sc = SigExtConfig.from_preset('it', '10k-60t')
data = get_test_data(lang='it', num_samples=30, skip_samples=sc.skip_samples)
sm, st = load_sigext_model(sc.model_id)
proc = preprocess_dataset(data, sm, st, lang='it')
del sm; clear_gpu_memory()
_, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', '8bit')
chain = create_summary_chain(pipe, get_summary_prompt('it', 'source_aware'))
inf_res = run_inference(proc, chain)
clear_gpu_memory()

In [ ]:
judge_results = {}
for jm in ['Qwen/Qwen2.5-14B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct']:
    _, _, jp = load_llm(jm, '8bit')
    jc = create_judge_chain(jp, get_judge_prompt('unified'))
    scores = {d: [] for d in ['faithfulness','completeness','conciseness','abstraction']}
    for r in inf_res:
        s = llm_judge_evaluate(r['source'], r['generated_summary'], r['reference'], jc)
        for d in scores: scores[d].append(s.get(d, 3))
    name = jm.split('/')[-1]
    judge_results[name] = {d: {'mean': float(np.mean(v)), 'std': float(np.std(v))} for d, v in scores.items()}
    clear_gpu_memory()
save_results({'ablation': 'judge_models', 'results': judge_results}, 'results/ablation_judge_models.json')